# Projeto BI: O Abismo da Saúde no Brasil
## Fase 2: Análise Exploratória de Dados (EDA) + Modelagem Dimensional

**Problema de Negócio:**  
> *"A distribuição de infraestrutura de saúde no Brasil é justa e proporcional à riqueza dos municípios, ou apenas reflete a desigualdade econômica?"*

**Fluxo deste notebook:**
1. Conexão e Leitura das Tabelas Raw (PostgreSQL)
2. Auditoria de Qualidade dos Dados
3. Análise de Distribuição e Outliers
4. Construção dos KPIs de Negócio
5. Análise de Correlação PIB × Cobertura de Saúde
6. Modelagem Dimensional (Star Schema) e Carga no DW

---

## Seção 0: Imports e Conexão com o PostgreSQL

In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

warnings.filterwarnings('ignore')

# Estilo dos gráficos — visual limpo e profissional
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 120

# Cria a pasta para salvar os gráficos (documentação e GitHub)
os.makedirs('graficos', exist_ok=True)

# Carrega as credenciais do .env
load_dotenv()

DB_URL = (
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASS')}"
    f"@{os.getenv('DB_HOST', 'localhost')}:{os.getenv('DB_PORT', '5432')}"
    f"/{os.getenv('DB_NAME')}"
)
engine = create_engine(DB_URL)

# Teste de conexão
with engine.connect() as conn:
    conn.execute(text('SELECT 1'))
print('✅ Conexão com PostgreSQL OK!')

---
## Seção 1: Leitura das Tabelas Raw

**Contexto:** Lemos as tabelas que foram carregadas pelo script `1_extracao_raw.py`.  
Nesta etapa ainda não fazemos nenhuma transformação — queremos ver os dados exatamente como chegaram da fonte.

In [ ]:
print('Lendo tabelas Raw do PostgreSQL...\n')

df_municipios    = pd.read_sql('SELECT * FROM raw_municipios',    engine)
df_populacao     = pd.read_sql('SELECT * FROM raw_populacao',     engine)
df_pib           = pd.read_sql('SELECT * FROM raw_pib',           engine)
df_leitos        = pd.read_sql('SELECT * FROM raw_leitos',        engine)
df_profissionais = pd.read_sql('SELECT * FROM raw_profissionais', engine)

tabelas = {
    'Municípios':    df_municipios,
    'População':     df_populacao,
    'PIB':           df_pib,
    'Leitos (CNES)': df_leitos,
    'Profissionais': df_profissionais,
}

for nome, df in tabelas.items():
    print(f'  {nome}: {df.shape[0]:,} linhas × {df.shape[1]} colunas')
    print(f'  Colunas: {list(df.columns)}\n')

---
## Seção 2: Auditoria de Qualidade dos Dados

**Por que fazer isso?**  
Antes de qualquer análise, precisamos entender o estado real dos dados.  
Dados nulos na coluna de código do município tornam impossível cruzar as tabelas.
Saber o percentual de nulos em cada coluna é o que justifica as **decisões de tratamento** que faremos na Seção 4 —
e é exatamente isso que os avaliadores esperam que você explique na apresentação.

In [ ]:
print('=== AUDITORIA DE QUALIDADE DOS DADOS RAW ===\n')

for nome, df in tabelas.items():
    print(f'--- {nome} ---')
    print(f'  Shape: {df.shape[0]:,} linhas × {df.shape[1]} colunas')

    nulos = df.isnull().sum()
    nulos_relevantes = nulos[nulos > 0]
    if len(nulos_relevantes) > 0:
        print('  Colunas com valores nulos:')
        for col, qtd in nulos_relevantes.items():
            pct = (qtd / len(df)) * 100
            print(f'    ⚠️  {col}: {qtd:,} nulos ({pct:.1f}%)')
    else:
        print('  ✅ Sem valores nulos')

    dups = df.duplicated().sum()
    print(f'  Linhas duplicadas: {dups:,}')
    print()

In [ ]:
# Inspecionar os tipos de dados e valores de exemplo de cada tabela
for nome, df in tabelas.items():
    print(f'=== {nome} — Amostra de 3 linhas ===')
    display(df.head(3))
    print()

---
## Seção 3: Análise de Distribuição e Outliers

**Por que fazer isso?**  
São Paulo tem mais de 75.000 leitos hospitalares. Um município pequeno do interior do Maranhão pode ter 12.  
Se trabalharmos com **números absolutos**, qualquer análise será completamente dominada pelas capitais,
tornando invisível a realidade do interior brasileiro — que é exatamente onde a desigualdade mais dói.

Esta seção documenta graficamente **por que precisamos normalizar os dados por habitante**.

In [ ]:
# ATENÇÃO: Os nomes de colunas abaixo podem variar conforme o dataset real.
# Ajuste col_leitos para o nome correto que aparecer no df_leitos.
# Após rodar a Seção 1, inspecione: print(df_leitos.columns.tolist())

# Identifique a coluna de quantidade de leitos
print('Colunas de df_leitos:', df_leitos.columns.tolist())

# Substitua 'QT_LEITO' abaixo pelo nome correto da coluna de quantidade
COL_LEITOS = 'QT_LEITO'  # <-- ajuste se necessário após ver as colunas acima

In [ ]:
# Agrega leitos por município para análise de distribuição
# (o CNES tem 1 linha por estabelecimento — precisamos somar por município)

# Identifique a coluna de código do município em df_leitos
print('Colunas disponíveis:', df_leitos.columns.tolist())

# Substitua 'CO_MUNICIPIO_GESTOR' pelo nome correto após ver as colunas
COL_COD_MUN_LEITOS = 'CO_MUNICIPIO_GESTOR'  # <-- ajuste se necessário

In [ ]:
# Agrupamento de leitos por município
leitos_por_mun = (
    df_leitos
    .groupby(COL_COD_MUN_LEITOS)[COL_LEITOS]
    .sum()
    .reset_index()
    .rename(columns={COL_COD_MUN_LEITOS: 'cod_municipio', COL_LEITOS: 'qt_leitos'})
)

print(f'Municípios com dados de leitos: {len(leitos_por_mun):,}')
print(leitos_por_mun.describe())

In [ ]:
# GRÁFICO 1: Distribuição — Linear vs Log
# Demonstra o problema dos outliers (São Paulo, Rio) com escala linear
# e como a escala logarítmica revela a distribuição real do interior.

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(leitos_por_mun['qt_leitos'], bins=60, ax=ax1, color='steelblue')
ax1.set_title('Distribuição de Leitos por Município\n(Escala Linear)', fontsize=13)
ax1.set_xlabel('Total de Leitos')
ax1.set_ylabel('Frequência (nº de municípios)')
ax1.annotate(
    'São Paulo domina\ncom ~75k leitos →',
    xy=(leitos_por_mun['qt_leitos'].max(), 1),
    xytext=(leitos_por_mun['qt_leitos'].quantile(0.75), 50),
    arrowprops=dict(arrowstyle='->', color='red'),
    fontsize=9, color='red'
)

sns.histplot(leitos_por_mun['qt_leitos'], bins=60, ax=ax2, color='steelblue', log_scale=(False, True))
ax2.set_title('Distribuição de Leitos por Município\n(Escala Logarítmica no Eixo Y)', fontsize=13)
ax2.set_xlabel('Total de Leitos')
ax2.set_ylabel('Frequência (escala log)')
ax2.text(
    0.6, 0.85,
    'A maioria dos municípios\ntem pouquíssimos leitos',
    transform=ax2.transAxes, fontsize=9,
    bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8)
)

plt.suptitle('EDA: Por que precisamos normalizar por habitante?', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('graficos/01_distribuicao_leitos.png', bbox_inches='tight')
plt.show()
print('💾 Gráfico salvo em graficos/01_distribuicao_leitos.png')

In [ ]:
# GRÁFICO 2: Top 10 e Bottom 10 municípios — valores absolutos
# Mostra que os rankings absolutos são dominados pelo tamanho da população,
# não pela qualidade da saúde pública.

top10 = leitos_por_mun.nlargest(10, 'qt_leitos')
bot10 = leitos_por_mun.nsmallest(10, 'qt_leitos')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

sns.barplot(data=top10, x='qt_leitos', y='cod_municipio', ax=ax1, palette='Reds_r')
ax1.set_title('Top 10 Municípios por Total de Leitos\n(Valores Absolutos — dominados pelas capitais)', fontsize=11)
ax1.set_xlabel('Total de Leitos')
ax1.set_ylabel('Código do Município')

sns.barplot(data=bot10, x='qt_leitos', y='cod_municipio', ax=ax2, palette='Blues_r')
ax2.set_title('Bottom 10 Municípios por Total de Leitos\n(Municípios com menos leitos do Brasil)', fontsize=11)
ax2.set_xlabel('Total de Leitos')
ax2.set_ylabel('Código do Município')

plt.suptitle('Ranking Absoluto × Ranking por Habitante — Resultados completamente diferentes', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('graficos/02_top_bottom_leitos.png', bbox_inches='tight')
plt.show()

---
## Seção 4: Transformação e Construção dos KPIs

**Por que fazer isso?**  
Esta é a etapa de **transformação do ETL** — equivalente às transformações que seriam feitas no Pentaho,
mas feitas em código Python de forma transparente e reproduzível.

Construiremos os dois KPIs principais:
- **Leitos por Mil Habitantes** — Métrica padrão da OMS. Mínimo recomendado: 3 leitos/mil hab.
- **Médicos por Mil Habitantes** — Mede a cobertura de recursos humanos em saúde.

**Decisões de tratamento que serão tomadas aqui (e explicadas):**
- Municípios sem código de identificação → **removidos** (impossível fazer JOIN)
- Municípios com leitos = 0 ou nulo → **mantidos com 0** (representam desertos de saúde — dado válido!)
- Municípios sem PIB registrado → **mantidos como NaN** (faremos análise separada)

In [ ]:
# ============================================================
# ATENÇÃO: Mapeamento de colunas
# Após rodar a Seção 1 e inspecionar as colunas reais,
# ajuste os nomes abaixo conforme necessário.
# ============================================================

# --- Colunas df_populacao ---
# Inspecione: print(df_populacao.columns.tolist())
COL_COD_MUN_POP  = 'CO_MUNICIPIO'  # código do município
COL_ANO_POP      = 'ANO'           # ano da estimativa
COL_POPULACAO    = 'POPULACAO'     # população estimada

# --- Colunas df_pib ---
# Inspecione: print(df_pib.columns.tolist())
COL_COD_MUN_PIB  = 'CO_MUNICIPIO'
COL_ANO_PIB      = 'ANO'
COL_PIB_TOTAL    = 'PIB'
COL_PIB_PERCAP   = 'PIB_PER_CAPITA'

# --- Colunas df_leitos ---
COL_COD_MUN_LEITOS = 'CO_MUNICIPIO_GESTOR'
COL_ANO_LEITOS     = 'ANO'
COL_LEITOS         = 'QT_LEITO'

# --- Colunas df_profissionais ---
# Inspecione: print(df_profissionais.columns.tolist())
COL_COD_MUN_PROF  = 'CO_MUNICIPIO'
COL_ANO_PROF      = 'ANO'
COL_CBO           = 'CBO'          # Código da ocupação (vamos filtrar médicos)
COL_QT_PROF       = 'QT_PROF'

# --- Colunas df_municipios ---
# Inspecione: print(df_municipios.columns.tolist())
COL_COD_MUN_DIM  = 'CO_MUNICIPIO'
COL_NOME_MUN     = 'NO_MUNICIPIO'
COL_UF           = 'SG_UF'
COL_REGIAO       = 'NO_REGIAO'

print('Mapeamento de colunas configurado!')
print('Se alguma coluna não existir, ajuste as variáveis acima conforme as colunas reais.')

In [ ]:
# ============================================================
# TRANSFORMAÇÃO 1: Agregação de Leitos
# O CNES tem 1 linha por estabelecimento de saúde.
# Precisamos somar todos os leitos de um mesmo município e ano.
# ============================================================

print('Agregando leitos por município e ano...')
leitos_agg = (
    df_leitos
    .groupby([COL_COD_MUN_LEITOS, COL_ANO_LEITOS])[COL_LEITOS]
    .sum()
    .reset_index()
    .rename(columns={
        COL_COD_MUN_LEITOS: 'cod_municipio',
        COL_ANO_LEITOS: 'ano',
        COL_LEITOS: 'qt_leitos'
    })
)
print(f'  Leitos agregados: {len(leitos_agg):,} registros (município × ano)\n')

# ============================================================
# TRANSFORMAÇÃO 2: Filtrar Médicos
# O CNES Profissionais tem todos os tipos: enfermeiros, técnicos, médicos...
# Filtraremos apenas os CBO que começam com '2251' (médicos).
# Código CBO 2251 = Médicos Clínicos (grande grupo de médicos no Brasil).
# Decisão: usamos o filtro mais amplo para não subregistrar municípios pequenos.
# ============================================================

print('Filtrando médicos e agregando por município e ano...')
medicos = df_profissionais[df_profissionais[COL_CBO].astype(str).str.startswith('2251')]

medicos_agg = (
    medicos
    .groupby([COL_COD_MUN_PROF, COL_ANO_PROF])[COL_QT_PROF]
    .sum()
    .reset_index()
    .rename(columns={
        COL_COD_MUN_PROF: 'cod_municipio',
        COL_ANO_PROF: 'ano',
        COL_QT_PROF: 'qt_medicos'
    })
)
print(f'  Médicos agregados: {len(medicos_agg):,} registros (município × ano)\n')

In [ ]:
# ============================================================
# TRANSFORMAÇÃO 3: Padronização das chaves de JOIN
# O código do município deve ter o mesmo formato em todas as tabelas.
# O IBGE usa 7 dígitos (ex: 3550308 = São Paulo)
# O DATASUS às vezes usa 6 dígitos (sem o dígito verificador).
# Aqui padronizamos tudo para string de 6 dígitos (truncando se necessário).
# ============================================================

print('Padronizando códigos de município...')

def padronizar_cod_municipio(df, col):
    """Converte o código do município para string de 6 dígitos."""
    df = df.copy()
    df[col] = df[col].astype(str).str.strip().str[:6]
    return df

pop = padronizar_cod_municipio(df_populacao.copy(), COL_COD_MUN_POP)
pop = pop.rename(columns={COL_COD_MUN_POP: 'cod_municipio', COL_ANO_POP: 'ano', COL_POPULACAO: 'populacao'})

pib = padronizar_cod_municipio(df_pib.copy(), COL_COD_MUN_PIB)
pib = pib.rename(columns={COL_COD_MUN_PIB: 'cod_municipio', COL_ANO_PIB: 'ano',
                           COL_PIB_TOTAL: 'pib_total', COL_PIB_PERCAP: 'pib_per_capita'})

leitos_agg  = padronizar_cod_municipio(leitos_agg, 'cod_municipio')
medicos_agg = padronizar_cod_municipio(medicos_agg, 'cod_municipio')

dim_mun = padronizar_cod_municipio(df_municipios.copy(), COL_COD_MUN_DIM)
dim_mun = dim_mun.rename(columns={
    COL_COD_MUN_DIM: 'cod_municipio',
    COL_NOME_MUN: 'nome_municipio',
    COL_UF: 'uf',
    COL_REGIAO: 'regiao'
})

print('Padronização concluída!')

In [ ]:
# ============================================================
# TRANSFORMAÇÃO 4: Merge Central (JOIN entre todas as tabelas)
# A população é a tabela base (tem um registro por município/ano).
# Fazemos LEFT JOIN para manter todos os municípios, mesmo sem dados de saúde.
# ============================================================

print('Executando o merge central (JOIN entre todas as tabelas)...')

df_final = (
    pop
    [['cod_municipio', 'ano', 'populacao']]
    .merge(leitos_agg  [['cod_municipio', 'ano', 'qt_leitos']],   on=['cod_municipio', 'ano'], how='left')
    .merge(medicos_agg [['cod_municipio', 'ano', 'qt_medicos']],  on=['cod_municipio', 'ano'], how='left')
    .merge(pib         [['cod_municipio', 'ano', 'pib_total', 'pib_per_capita']], on=['cod_municipio', 'ano'], how='left')
    .merge(dim_mun     [['cod_municipio', 'nome_municipio', 'uf', 'regiao']], on='cod_municipio', how='left')
)

# Preenchimento de nulos:
# Leitos e médicos nulos significam ZERO (o município não tem registros no CNES).
# Isso representa um deserto de saúde — dado válido e importante!
df_final['qt_leitos']  = df_final['qt_leitos'].fillna(0)
df_final['qt_medicos'] = df_final['qt_medicos'].fillna(0)

print(f'  Registros no dataset final: {len(df_final):,}')
print(f'  Municípios únicos: {df_final["cod_municipio"].nunique():,}')
print(f'  Anos disponíveis: {sorted(df_final["ano"].unique())}')
print(f'\n  Nulos restantes:')
print(df_final.isnull().sum())

In [ ]:
# ============================================================
# CONSTRUÇÃO DOS KPIs
# ============================================================

# KPI 1: Leitos por Mil Habitantes
# Fórmula: (total de leitos / população) × 1000
# Referência OMS: mínimo de 3 leitos por mil habitantes
df_final['leitos_por_mil_hab'] = np.where(
    df_final['populacao'] > 0,
    (df_final['qt_leitos'] / df_final['populacao']) * 1000,
    np.nan
)

# KPI 2: Médicos por Mil Habitantes
df_final['medicos_por_mil_hab'] = np.where(
    df_final['populacao'] > 0,
    (df_final['qt_medicos'] / df_final['populacao']) * 1000,
    np.nan
)

# KPI 3: Classificação OMS
# Categoriza municípios por faixa de cobertura de saúde.
# Útil para criar segmentações no Power BI.
df_final['status_oms'] = pd.cut(
    df_final['leitos_por_mil_hab'],
    bins=[-0.01, 1, 3, float('inf')],
    labels=['Crítico (<1 leito/mil)', 'Alerta (1–3 leitos/mil)', 'Adequado (>3 leitos/mil)']
)

print('KPIs calculados com sucesso!')
print('\nEstatísticas descritivas — Leitos por Mil Habitantes:')
print(df_final['leitos_por_mil_hab'].describe())

print('\nDistribuição por Status OMS:')
print(df_final['status_oms'].value_counts())

---
## Seção 5: Análise de Correlação — A Ilusão do PIB

**Por que fazer isso?**  
Esta é a análise central do nosso storytelling.  
Se a relação entre riqueza econômica e cobertura de saúde fosse linear, a política pública seria simples: 
"basta crescer economicamente". Mas os dados mostram algo bem mais complexo.

Municípios com PIB alto e cobertura baixa **revelam uma falha de política pública** —  
a riqueza gerada no município não está sendo convertida em saúde para a população local.

In [ ]:
# Filtramos o ano mais recente para a análise de correlação
# (evitamos misturar tendências temporais no scatter)
ano_recente = df_final['ano'].max()
df_scatter = df_final[
    (df_final['ano'] == ano_recente) &
    (df_final['pib_per_capita'].notna()) &
    (df_final['leitos_por_mil_hab'].notna()) &
    (df_final['populacao'] > 5000)   # remove municípios muito pequenos (dados instáveis)
].copy()

print(f'Municípios no scatter (ano {ano_recente}, pop > 5.000): {len(df_scatter):,}')

# Correlação de Pearson entre PIB per capita e Leitos por mil hab
corr = df_scatter['pib_per_capita'].corr(df_scatter['leitos_por_mil_hab'])
print(f'Correlação de Pearson (PIB per capita × Leitos/mil hab): {corr:.3f}')
print()
if abs(corr) < 0.3:
    print('📌 ACHADO: Correlação fraca — riqueza econômica NÃO garante cobertura de saúde!')
elif abs(corr) < 0.6:
    print('📌 ACHADO: Correlação moderada — relação existe, mas com muitas exceções importantes.')
else:
    print('📌 ACHADO: Correlação forte — municípios mais ricos tendem a ter mais leitos.')

In [ ]:
# GRÁFICO 3: Scatter PIB per capita × Leitos por Mil Hab
# Este é o gráfico mais importante do projeto — evidencia a hipótese central.

regioes = df_scatter['regiao'].unique()
palette = dict(zip(regioes, sns.color_palette('tab10', len(regioes))))

fig, ax = plt.subplots(figsize=(14, 8))

for regiao, grupo in df_scatter.groupby('regiao'):
    ax.scatter(
        x=grupo['pib_per_capita'],
        y=grupo['leitos_por_mil_hab'],
        label=regiao,
        alpha=0.5, s=20,
        color=palette.get(regiao, 'gray')
    )

# Linha do mínimo recomendado pela OMS
ax.axhline(y=3, color='red', linestyle='--', linewidth=1.5, label='Mínimo OMS (3 leitos/mil hab)')

ax.set_xlabel('PIB per capita (R$)', fontsize=12)
ax.set_ylabel('Leitos SUS por Mil Habitantes', fontsize=12)
ax.set_title(
    f'A Ilusão do PIB: Riqueza Econômica ≠ Cobertura de Saúde\n'
    f'Correlação de Pearson: r = {corr:.3f} — Ano {ano_recente} — Municípios com pop. > 5.000',
    fontsize=13, fontweight='bold'
)
ax.legend(title='Região', fontsize=9)

# Destaque: municípios com alto PIB e baixa cobertura (quadrante problemático)
p75_pib    = df_scatter['pib_per_capita'].quantile(0.75)
mediana_leitos = df_scatter['leitos_por_mil_hab'].median()
ax.axvspan(p75_pib, df_scatter['pib_per_capita'].max(), alpha=0.05, color='red')
ax.text(p75_pib * 1.01, 0.2, 'Municípios ricos com\nbaixa cobertura = Gap de política pública',
        fontsize=8, color='darkred', style='italic')

plt.tight_layout()
plt.savefig('graficos/03_scatter_pib_leitos.png', bbox_inches='tight')
plt.show()
print('💾 Gráfico salvo em graficos/03_scatter_pib_leitos.png')

In [ ]:
# GRÁFICO 4: Capital vs Interior — Fuga de Cérebros
# Cria uma flag para municípios que são capitais estaduais

CAPITAIS = [
    'Porto Velho', 'Rio Branco', 'Manaus', 'Boa Vista', 'Belém',
    'Macapá', 'Palmas', 'São Luís', 'Teresina', 'Fortaleza',
    'Natal', 'João Pessoa', 'Recife', 'Maceió', 'Aracaju', 'Salvador',
    'Belo Horizonte', 'Vitória', 'Rio de Janeiro', 'São Paulo',
    'Curitiba', 'Florianópolis', 'Porto Alegre',
    'Campo Grande', 'Cuiabá', 'Goiânia', 'Brasília'
]

df_scatter['tipo'] = df_scatter['nome_municipio'].apply(
    lambda x: 'Capital' if str(x) in CAPITAIS else 'Interior'
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

medicos_tipo = df_scatter.groupby('tipo')['medicos_por_mil_hab'].mean().reset_index()
sns.barplot(data=medicos_tipo, x='tipo', y='medicos_por_mil_hab', ax=ax1, palette=['#e74c3c', '#3498db'])
ax1.set_title('Médicos por Mil Hab: Capital vs Interior\n(Média Nacional)', fontsize=12)
ax1.set_xlabel('')
ax1.set_ylabel('Médicos por Mil Habitantes')
for bar in ax1.patches:
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.05,
             f'{bar.get_height():.2f}', ha='center', va='bottom', fontweight='bold')

leitos_tipo = df_scatter.groupby('tipo')['leitos_por_mil_hab'].mean().reset_index()
sns.barplot(data=leitos_tipo, x='tipo', y='leitos_por_mil_hab', ax=ax2, palette=['#e74c3c', '#3498db'])
ax2.axhline(y=3, color='red', linestyle='--', linewidth=1.5, label='Mínimo OMS')
ax2.set_title('Leitos por Mil Hab: Capital vs Interior\n(Média Nacional)', fontsize=12)
ax2.set_xlabel('')
ax2.set_ylabel('Leitos por Mil Habitantes')
ax2.legend()
for bar in ax2.patches:
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.05,
             f'{bar.get_height():.2f}', ha='center', va='bottom', fontweight='bold')

plt.suptitle('A Fuga de Cérebros: Concentração de Recursos de Saúde nas Capitais', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('graficos/04_capital_vs_interior.png', bbox_inches='tight')
plt.show()
print('💾 Gráfico salvo em graficos/04_capital_vs_interior.png')

In [ ]:
# TABELA: Top 10 Municípios que mais precisam de investimento
# (alto PIB, baixa cobertura — o GAP de política pública)

gap = df_scatter[
    (df_scatter['pib_per_capita'] > df_scatter['pib_per_capita'].quantile(0.5)) &  # PIB acima da mediana
    (df_scatter['leitos_por_mil_hab'] < 2)  # leitos abaixo de 2/mil hab
].copy()

gap_sorted = (
    gap
    .nlargest(10, 'pib_per_capita')
    [['nome_municipio', 'uf', 'regiao', 'pib_per_capita', 'leitos_por_mil_hab', 'medicos_por_mil_hab', 'populacao']]
    .rename(columns={
        'nome_municipio': 'Município', 'uf': 'UF', 'regiao': 'Região',
        'pib_per_capita': 'PIB per Capita (R$)',
        'leitos_por_mil_hab': 'Leitos/Mil Hab',
        'medicos_por_mil_hab': 'Médicos/Mil Hab',
        'populacao': 'População'
    })
)

print('🚨 TOP 10 MUNICÍPIOS COM MAIOR GAP (PIB alto, cobertura baixa):')
print('   → Estes são os municípios prioritários para investimento em saúde pública')
display(gap_sorted)

---
## Seção 6: Modelagem Dimensional — Star Schema

**Por que fazer isso?**  
O Star Schema é o modelo de dados padrão para Data Warehouses e ferramentas de BI como o Power BI.  
Ao separar os dados em tabelas **Fato** (métricas numéricas) e **Dimensões** (contexto descritivo),
criamos um modelo otimizado para consultas analíticas — exatamente o que o Power BI vai usar.

**Estrutura:**
```
dim_municipio (cod_municipio) ←→ fato_saude_economia (cod_municipio)
dim_tempo     (id_tempo)      ←→ fato_saude_economia (ano)
```

In [ ]:
print('=== FASE 2: CONSTRUÇÃO DO STAR SCHEMA (Data Warehouse) ===\n')

# --- dim_municipio ---
# Tabela de dimensão com os atributos descritivos dos municípios.
# cod_municipio é a Primary Key que vai fazer JOIN com a Fato.
dim_municipio = (
    dim_mun
    [['cod_municipio', 'nome_municipio', 'uf', 'regiao']]
    .drop_duplicates(subset='cod_municipio')
    .reset_index(drop=True)
)
print(f'dim_municipio: {len(dim_municipio):,} municípios')

# --- dim_tempo ---
# Tabela de dimensão temporal — simples para este projeto (só o ano).
# Em projetos maiores, incluiria trimestre, semestre, etc.
anos_disponiveis = sorted(df_final['ano'].dropna().unique())
dim_tempo = pd.DataFrame({'id_tempo': anos_disponiveis, 'ano': anos_disponiveis})
print(f'dim_tempo: {len(dim_tempo)} anos (de {min(anos_disponiveis)} a {max(anos_disponiveis)})')

# --- fato_saude_economia ---
# Tabela Fato: contém as métricas numéricas e as Foreign Keys para as dimensões.
# Granularidade: 1 registro por município por ano.
colunas_fato = [
    'cod_municipio', 'ano',          # Foreign Keys
    'populacao',                      # Medida base
    'pib_total', 'pib_per_capita',    # Economia
    'qt_leitos', 'qt_medicos',        # Infraestrutura absoluta
    'leitos_por_mil_hab',             # KPI 1 (OMS)
    'medicos_por_mil_hab',            # KPI 2
    'status_oms'                      # KPI 3 (Classificação)
]
fato = (
    df_final[colunas_fato]
    .dropna(subset=['cod_municipio', 'ano'])  # Remove linhas sem chaves de JOIN
    .reset_index(drop=True)
)
print(f'fato_saude_economia: {len(fato):,} registros')
print(f'  (granularidade: município × ano)')

In [ ]:
# ============================================================
# CARGA NO DATA WAREHOUSE (schema: dw)
# ============================================================
# As tabelas vão para o schema 'dw' — separado do schema 'public'
# que tem os dados Raw. Isso simula a separação de camadas
# que existe em qualquer Data Lake corporativo.

print('Carregando tabelas no Data Warehouse (schema: dw)...\n')

dim_municipio.to_sql('dim_municipio', engine, if_exists='replace', schema='dw', index=False)
print(f'  ✅ dw.dim_municipio: {len(dim_municipio):,} registros')

dim_tempo.to_sql('dim_tempo', engine, if_exists='replace', schema='dw', index=False)
print(f'  ✅ dw.dim_tempo: {len(dim_tempo)} registros')

fato.to_sql('fato_saude_economia', engine, if_exists='replace', schema='dw',
            index=False, chunksize=10_000)
print(f'  ✅ dw.fato_saude_economia: {len(fato):,} registros')

print('\n🎉 STAR SCHEMA CRIADO COM SUCESSO NO POSTGRESQL!')
print('   Schema: dw | Tabelas: dim_municipio, dim_tempo, fato_saude_economia')

In [ ]:
# ============================================================
# VERIFICAÇÃO FINAL DO DATA WAREHOUSE
# ============================================================
# Consultas SQL diretas no banco para confirmar que tudo foi carregado.

print('=== VERIFICAÇÃO DO DATA WAREHOUSE ===\n')

queries_verificacao = {
    'Total de registros na Fato': 'SELECT COUNT(*) FROM dw.fato_saude_economia',
    'Período coberto': 'SELECT MIN(ano), MAX(ano) FROM dw.fato_saude_economia',
    'Municípios únicos': 'SELECT COUNT(DISTINCT cod_municipio) FROM dw.fato_saude_economia',
    'Média de Leitos por Mil Hab': 'SELECT ROUND(AVG(leitos_por_mil_hab)::numeric, 2) FROM dw.fato_saude_economia',
    '% Municípios Abaixo do Mínimo OMS (< 3)': '''
        SELECT ROUND(100.0 * COUNT(*) FILTER (WHERE leitos_por_mil_hab < 3) / COUNT(*), 1) 
        FROM dw.fato_saude_economia
    '''
}

with engine.connect() as conn:
    for descricao, query in queries_verificacao.items():
        resultado = conn.execute(text(query)).fetchone()
        print(f'  {descricao}: {resultado[0]}')

print('\n✅ Data Warehouse verificado e pronto para o Power BI!')
print('   Próximo passo: abra o Power BI Desktop e conecte ao PostgreSQL (schema: dw)')